Mount drive:

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


Dẫn đến đường dẫn:

In [3]:
# Acc phụ:
%cd /content/drive/MyDrive/BASNet
# Acc chính:
# %cd /content/drive/MyDrive/PBL4/BASNet
!ls
!ls train_data/DUTS/DUTS-TR/

/content/drive/.shortcut-targets-by-id/15jFIt9F6jfLRWDXNZuJdrtYqDk3A6CCT/BASNet
basnet_test.py	       demo	__pycache__   README.md~    validation_data
BASNet_training.ipynb  figures	pytorch_iou   saved_models
basnet_train.py        LICENSE	pytorch_ssim  test_data
data_loader.py	       model	README.md     train_data
DUTS-TR-Image  DUTS-TR-Mask


Đoạn code train:

In [ ]:
# ================================================================
#  BASNet Training — Fixed
#  Chạy trong Colab cell (sau khi mount Drive và cd đúng thư mục)
# ================================================================
import math, os, glob, random, gc
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.amp
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm
from scipy.ndimage import (distance_transform_edt,
                           binary_erosion, binary_dilation,
                           generate_binary_structure)

from data_loader import (RescaleT, RandomCrop, CenterCrop,
                         RandomHorizontalFlip, RandomVerticalFlip,
                         ColorJitter, ToTensorLab, SalObjDataset)
from model import BASNet
import pytorch_ssim
import pytorch_iou

torch.cuda.empty_cache()

# ================================================================
#  EMA
# ================================================================
class ModelEMA:
    def __init__(self, model, decay=0.999):
        self.decay  = decay
        self.shadow = {k: v.detach().float().clone()
                       for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            self.shadow[k].mul_(self.decay).add_(
                v.detach().float(), alpha=1.0 - self.decay)

    def swap(self, model):
        original = {k: v.detach().clone()
                    for k, v in model.state_dict().items()}
        model.load_state_dict(
            {k: v.to(next(model.parameters()).device)
             for k, v in self.shadow.items()})
        return original

    def restore(self, model, original_state):
        model.load_state_dict(original_state)

    def state_dict(self):  return self.shadow
    def load_state_dict(self, s): self.shadow = {k: v.float() for k,v in s.items()}


# ================================================================
#  CHECKPOINT
# ================================================================
CHECKPOINT_PATH = "./saved_models/basnet_bsi/checkpoint_resume.pth"

def save_checkpoint(epoch, net, ema, optimizer, scheduler, scaler,
                    best_val_loss, best_epoch,
                    train_losses, val_losses, lr_list,
                    mae_list, sm_list, em_list, wfm_list, bfm_list):
    os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
    tmp_path = CHECKPOINT_PATH + ".tmp"
    torch.save({
        "epoch": epoch, "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "model_state":  net.state_dict(),
        "ema_state":    ema.state_dict(),
        "optim_state":  optimizer.state_dict(),
        "sched_state":  scheduler.state_dict(),
        "scaler_state": scaler.state_dict(),
        "train_losses": train_losses, "val_losses": val_losses,
        "lr_list": lr_list, "mae_list": mae_list,
        "sm_list": sm_list, "em_list": em_list,
        "wfm_list": wfm_list, "bfm_list": bfm_list,
    }, tmp_path)
    import shutil
    shutil.move(tmp_path, CHECKPOINT_PATH)   # atomic write
    print(f"  [Checkpoint] Saved epoch {epoch+1} → {CHECKPOINT_PATH}")

def load_checkpoint(net, ema, optimizer, scheduler, scaler):
    if not os.path.exists(CHECKPOINT_PATH):
        print("[Checkpoint] Not found — training from scratch.")
        return 0, float('inf'), 0, [], [], [], [], [], [], [], []
    print(f"[Checkpoint] Found: {CHECKPOINT_PATH}")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
    net.load_state_dict(ckpt["model_state"])
    if "ema_state" in ckpt: ema.load_state_dict(ckpt["ema_state"])
    optimizer.load_state_dict(ckpt["optim_state"])
    scheduler.load_state_dict(ckpt["sched_state"])
    if "scaler_state" in ckpt: scaler.load_state_dict(ckpt["scaler_state"])
    se = ckpt["epoch"] + 1
    print(f"  → Resume epoch {se+1}/{epoch_num} | best: {ckpt['best_val_loss']:.6f} (ep{ckpt['best_epoch']})")
    return (se, ckpt["best_val_loss"], ckpt["best_epoch"],
            ckpt["train_losses"], ckpt["val_losses"], ckpt["lr_list"],
            ckpt["mae_list"], ckpt["sm_list"], ckpt["em_list"],
            ckpt["wfm_list"], ckpt["bfm_list"])


# ================================================================
#  METRICS
# ================================================================
def compute_mae(pred, gt):
    p = pred.squeeze().cpu().float().numpy().astype(np.float64)
    g = gt.squeeze().cpu().float().numpy().astype(np.float64)
    return float(np.mean(np.abs(p - g)))

def _ssim_region(pred, gt):
    x, y   = pred.mean(), gt.mean()
    sx     = ((pred-x)**2).mean(); sy = ((gt-y)**2).mean()
    sxy    = ((pred-x)*(gt-y)).mean()
    num    = 4*x*y*sxy
    den    = (x**2+y**2)*(sx+sy)+1e-8
    if num == 0: return 1.0 if den==0 else 0.0
    return float(num/den)

def _centroid(gt):
    r, c = gt.shape
    if gt.sum()==0: return c//2, r//2
    jj, ii = np.meshgrid(np.arange(c), np.arange(r))
    t = gt.sum()
    return int((jj*gt).sum()/t), int((ii*gt).sum()/t)

def _divide(a, X, Y): return a[:Y,:X], a[:Y,X:], a[Y:,:X], a[Y:,X:]

def _So(p, g):
    u = g.mean()
    return u*_ssim_region(p*g,g) + (1-u)*_ssim_region((1-p)*(1-g),1-g)

def _Sr(p, g):
    X,Y   = _centroid(g)
    total = g.shape[0]*g.shape[1]
    return sum(gp.size/total*_ssim_region(pp,gp)
               for gp,pp in zip(_divide(g,X,Y),_divide(p,X,Y)))

def compute_smeasure(pred, gt, alpha=0.5):
    p = pred.squeeze().cpu().float().numpy().astype(np.float64)
    g = (gt.squeeze().cpu().float().numpy()>=0.5).astype(np.float64)
    y = g.mean()
    if   y==0: s = 1.0-p.mean()
    elif y==1: s = p.mean()
    else:      s = alpha*_So(p,g)+(1-alpha)*_Sr(p,g)
    return float(max(0.0,s))

def compute_emeasure(pred, gt):
    p = pred.squeeze().cpu().float().numpy().astype(np.float64)
    g = (gt.squeeze().cpu().float().numpy()>=0.5).astype(np.float64)
    if   g.max()==0: return float((1-p).mean())
    elif g.min()==1: return float(p.mean())
    dp, dg = p-p.mean(), g-g.mean()
    align  = 2*dp*dg/(dp**2+dg**2+1e-8)
    return float(((align+1)**2/4).mean())

def compute_wfmeasure(pred, gt, beta_sq=1.0):
    p = pred.squeeze().cpu().float().numpy().astype(np.float64)
    g = (gt.squeeze().cpu().float().numpy()>=0.5).astype(np.float64)
    if g.max()==0: return 0.0
    Do = distance_transform_edt(1-g); Di = distance_transform_edt(g)
    if Do.max()>0: Do=Do/Do.max()
    if Di.max()>0: Di=Di/Di.max()
    w  = np.where(g>0.5, 1+5*np.abs(Di-0.5), 1+5*Do)
    TP = (p*g*w).sum(); FP = (p*(1-g)*w).sum(); FN = ((1-p)*g*w).sum()
    P  = TP/(TP+FP+1e-8); R = TP/(TP+FN+1e-8)
    return float((1+beta_sq)*P*R/(beta_sq*P+R+1e-8))

def compute_boundary_fmeasure(pred, gt, threshold=0.5, beta_sq=0.3, tolerance=3):
    pb = (pred.squeeze().cpu().float().numpy()>=threshold).astype(np.uint8)
    gb = (gt.squeeze().cpu().float().numpy()>=0.5).astype(np.uint8)
    s  = generate_binary_structure(2,1)
    pb_bd = pb ^ binary_erosion(pb,s).astype(np.uint8)
    gb_bd = gb ^ binary_erosion(gb,s).astype(np.uint8)
    pb_d  = binary_dilation(pb_bd, iterations=tolerance)
    gb_d  = binary_dilation(gb_bd, iterations=tolerance)
    ps,gs = pb_bd.sum(), gb_bd.sum()
    P = float((pb_bd&gb_d).sum())/(ps+1e-8) if ps>0 else 0.0
    R = float((gb_bd&pb_d).sum())/(gs+1e-8) if gs>0 else 0.0
    return float((1+beta_sq)*P*R/(beta_sq*P+R+1e-8))


# ================================================================
#  SETUP
# ================================================================
torch.backends.cudnn.benchmark     = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.enabled       = True
torch.backends.cudnn.allow_tf32    = True
torch.backends.cuda.matmul.allow_tf32 = True
# expandable_segments không hỗ trợ trên Windows — đã bỏ

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# ================================================================
#  LOSS
#  FIX 1: BCELoss dùng reduction='mean' thay vì size_average=True (deprecated)
# ================================================================
bce_loss  = nn.BCELoss(reduction='mean')
ssim_loss = pytorch_ssim.SSIM(window_size=9, size_average=True)
iou_loss  = pytorch_iou.IOU(size_average=True)

LOSS_W_BCE  = 1.0
LOSS_W_SSIM = 1.0
LOSS_W_IOU  = 1.0
OUTPUT_WEIGHTS = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

def bce_ssim_iou_loss(pred, target):
    return (bce_loss(pred, target)          * LOSS_W_BCE
            + (1 - ssim_loss(pred, target)) * LOSS_W_SSIM
            + iou_loss(pred, target)        * LOSS_W_IOU)

def muti_loss_fusion(d0,d1,d2,d3,d4,d5,d6,d7, labels_v):
    outs   = [d0,d1,d2,d3,d4,d5,d6,d7]
    losses = [bce_ssim_iou_loss(o, labels_v) for o in outs]
    total  = sum(w*l for w,l in zip(OUTPUT_WEIGHTS, losses))
    return losses[0], total


# ================================================================
#  CONFIG  (tối ưu cho GPU 4GB — RTX 3050 / GTX 1650 / etc.)
# ================================================================
EFFECTIVE_BATCH_SIZE = 16   # giữ nguyên effective batch để training ổn định
batch_size_train     = 1    # GPU 4GB: chỉ fit được batch=1 ở 256×256
batch_size_val       = 1

assert EFFECTIVE_BATCH_SIZE % batch_size_train == 0
accumulation_steps = EFFECTIVE_BATCH_SIZE // batch_size_train  # = 16

epoch_num  = 30
train_num  = 6000
val_num    = 1500
SIZE_TRAIN = 256


# ================================================================
#  DATA
# ================================================================
data_dir      = './train_data/'
tra_image_dir = 'DUTS/DUTS-TR/DUTS-TR-Image/'
tra_label_dir = 'DUTS/DUTS-TR/DUTS-TR-Mask/'
model_dir     = "./saved_models/basnet_bsi/"

all_img = glob.glob(data_dir + tra_image_dir + '*.jpg')
all_lbl = [os.path.join(data_dir, tra_label_dir,
           os.path.splitext(os.path.basename(p))[0]+'.png')
           for p in all_img]
combined = list(zip(all_img, all_lbl))
random.seed(42); random.shuffle(combined)
all_img, all_lbl = map(list, zip(*combined))

train_dataset = SalObjDataset(
    img_name_list=all_img[:train_num],
    lbl_name_list=all_lbl[:train_num],
    transform=transforms.Compose([
        RescaleT(SIZE_TRAIN),
        RandomCrop(224),
        RandomHorizontalFlip(p=0.5),
        RandomVerticalFlip(p=0.2),
        ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.4),
        ToTensorLab(flag=0),
    ])
)
val_dataset = SalObjDataset(
    img_name_list=all_img[train_num:train_num+val_num],
    lbl_name_list=all_lbl[train_num:train_num+val_num],
    transform=transforms.Compose([
        RescaleT(SIZE_TRAIN),
        CenterCrop(224),
        ToTensorLab(flag=0),
    ])
)
train_loader = DataLoader(train_dataset, batch_size=batch_size_train,
                          shuffle=True,  num_workers=0, pin_memory=False,  # num_workers=0 trên Windows
                          drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size_val,
                          shuffle=False, num_workers=0, pin_memory=False,
                          drop_last=False)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Effective batch: {EFFECTIVE_BATCH_SIZE} "
      f"(bs={batch_size_train} × accum={accumulation_steps})")
print(f"Loss weights: BCE×{LOSS_W_BCE} | SSIM×{LOSS_W_SSIM} | IOU×{LOSS_W_IOU}")


# ================================================================
#  MODEL / EMA / OPTIMIZER / SCHEDULER
# ================================================================
net = BASNet(3, 1).to(device)
print(f"Model on: {next(net.parameters()).device}")

ema = ModelEMA(net, decay=0.999)

optimizer = optim.Adam(net.parameters(), lr=5e-5,
                       betas=(0.9,0.999), eps=1e-8, weight_decay=1e-4)

# FIX 2: T_max = tổng số lần optimizer.step() thực sự xảy ra
# = số batch thực / accumulation_steps × epoch_num
steps_per_epoch = math.ceil(len(train_loader) / accumulation_steps)
total_steps     = steps_per_epoch * epoch_num
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=1e-6)

scaler = torch.amp.GradScaler('cuda')


# ================================================================
#  LOAD CHECKPOINT
# ================================================================
(start_epoch, best_val_loss, best_epoch,
 train_losses, val_losses, lr_list,
 mae_list, sm_list, em_list, wfm_list, bfm_list) = load_checkpoint(
    net, ema, optimizer, scheduler, scaler)

if start_epoch >= epoch_num:
    print(f"Done ({epoch_num} ep). Delete checkpoint to retrain.")
    raise SystemExit

print(f"\n--- Training from epoch {start_epoch+1}/{epoch_num} ---")
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    free, tot = torch.cuda.mem_get_info()
    print(f"GPU: {free/1024**3:.1f}GB free / {tot/1024**3:.1f}GB total")

best_wfm = max(wfm_list) if wfm_list else 0.0
best_mae = min(mae_list)  if mae_list else float('inf')


# ================================================================
#  TRAINING LOOP
# ================================================================
for epoch in range(start_epoch, epoch_num):

    # ── TRAIN ───────────────────────────────────────────────
    net.train()
    optimizer.zero_grad()

    epoch_train_loss = running_loss = running_tar = 0.0
    n_iter = 0
    total_batches = len(train_loader)

    train_bar = tqdm(train_loader,
                     desc=f"Epoch {epoch+1}/{epoch_num} [train]", leave=True)
    for i, data in enumerate(train_bar):
        n_iter += 1
        is_last     = (i+1 == total_batches)
        should_step = ((i+1) % accumulation_steps == 0) or is_last

        inputs = data['image'].float().to(device)
        labels = data['label'].float().to(device)

        # FIX 3: forward trong autocast, backward và loss tính ngoài
        with torch.amp.autocast('cuda', dtype=torch.float16):
            d0,d1,d2,d3,d4,d5,d6,d7 = net(inputs)

        # Cast về float32 trước khi tính loss (SSIM/IoU cần fp32)
        d0,d1,d2,d3 = d0.float(),d1.float(),d2.float(),d3.float()
        d4,d5,d6,d7 = d4.float(),d5.float(),d6.float(),d7.float()

        loss_d0, loss = muti_loss_fusion(d0,d1,d2,d3,d4,d5,d6,d7, labels)

        # FIX 4: chia đều cho accumulation_steps (cố định), không dùng window_size
        loss_norm = loss / accumulation_steps

        running_loss     += loss.item()
        running_tar      += loss_d0.item()
        epoch_train_loss += loss.item()

        # FIX 5: backward TRƯỚC khi del tensor
        scaler.scale(loss_norm).backward()

        del d0,d1,d2,d3,d4,d5,d6,d7,loss_d0,loss,loss_norm
        inputs = labels = None
        torch.cuda.empty_cache()  # giải phóng memory mỗi batch (quan trọng với GPU 4GB)

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            ema.update(net)
            torch.cuda.empty_cache()

        if (i+1) % 50 == 0:
            train_bar.set_postfix({
                "loss": f"{running_loss/n_iter:.4f}",
                "d0":   f"{running_tar/n_iter:.4f}",
                "lr":   f"{optimizer.param_groups[0]['lr']:.2e}",
            })

    epoch_train_loss /= total_batches
    train_losses.append(epoch_train_loss)

    # ── VALIDATION (EMA weights) ─────────────────────────────
    original_state = ema.swap(net)
    net.eval()

    val_loss = val_tar = 0.0
    val_iter = 0
    total_mae=total_sm=total_em=total_wfm=total_bfm = 0.0
    total_samples = 0

    val_bar = tqdm(val_loader,
                   desc=f"Epoch {epoch+1}/{epoch_num} [val/EMA]", leave=True)
    with torch.no_grad():
        for i, data in enumerate(val_bar):
            val_iter += 1
            inputs = data['image'].float().to(device)
            labels = data['label'].float().to(device)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                d0,_,_,_,_,_,_,_ = net(inputs)

            # FIX 6: cast về float32 để tính metrics chính xác
            pred = d0.float()

            # FIX 7: val loss chỉ tính trên d0 (không nhân 8 lần)
            val_loss += bce_ssim_iou_loss(pred, labels).item()
            val_tar  += val_loss

            # Metrics tính ngoài autocast (float32, trên CPU)
            for b in range(pred.shape[0]):
                total_mae += compute_mae(pred[b], labels[b])
                total_sm  += compute_smeasure(pred[b], labels[b])
                total_em  += compute_emeasure(pred[b], labels[b])
                total_wfm += compute_wfmeasure(pred[b], labels[b])
                total_bfm += compute_boundary_fmeasure(pred[b], labels[b])
                total_samples += 1

            if (i+1) % 50 == 0:
                val_bar.set_postfix({
                    "MAE": f"{total_mae/total_samples:.4f}",
                    "Sm":  f"{total_sm /total_samples:.3f}",
                    "Em":  f"{total_em /total_samples:.3f}",
                    "wFm": f"{total_wfm/total_samples:.3f}",
                    "loss":f"{val_loss /val_iter:.4f}",
                })

    ema.restore(net, original_state)
    net.train()

    avg_val_loss = val_loss / val_iter
    avg_mae = total_mae/total_samples; avg_sm  = total_sm /total_samples
    avg_em  = total_em /total_samples; avg_wfm = total_wfm/total_samples
    avg_bfm = total_bfm/total_samples

    val_losses.append(avg_val_loss)
    mae_list.append(avg_mae); sm_list.append(avg_sm)
    em_list.append(avg_em);   wfm_list.append(avg_wfm); bfm_list.append(avg_bfm)

    current_lr = optimizer.param_groups[0]['lr']
    lr_list.append(current_lr)

    print(f"\n[Epoch {epoch+1}] Train={epoch_train_loss:.4f} | Val(EMA)={avg_val_loss:.4f}")
    print(f"  wFm={avg_wfm:.4f}  bFm={avg_bfm:.4f}  "
          f"MAE={avg_mae:.4f}(↓)  Sm={avg_sm:.4f}  Em={avg_em:.4f}")
    print(f"  LR → {current_lr:.2e}")

    torch.cuda.empty_cache(); gc.collect()
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.memory_allocated()/1024**3:.2f}GB used | "
              f"{torch.cuda.memory_reserved()/1024**3:.2f}GB reserved")

    # ── Save ────────────────────────────────────────────────
    os.makedirs(model_dir, exist_ok=True)
    dev = next(net.parameters()).device
    ema_weights = {k: v.to(dev) for k,v in ema.state_dict().items()}

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss; best_epoch = epoch+1
        torch.save(ema_weights, model_dir+"basnet_best_valloss.pth")
        print("  ✓ Saved best val loss [EMA]")
    if len(sm_list)==1 or avg_sm>=max(sm_list[:-1]):
        torch.save(ema_weights, model_dir+"basnet_epoch_best_sm.pth")
        print("  ✓ Saved best Sm [EMA]")
    if len(wfm_list)==1 or avg_wfm>=max(wfm_list[:-1]):
        best_wfm = avg_wfm
        torch.save(ema_weights, model_dir+"basnet_best_wfm.pth")
        print("  ✓ Saved best WFm [EMA]")
    if avg_mae < best_mae:
        best_mae = avg_mae
        torch.save(ema_weights, model_dir+"basnet_best_mae.pth")
        print("  ✓ Saved best MAE [EMA]")

    print(f"  Best → loss={best_val_loss:.6f}(ep{best_epoch}) "
          f"WFm={best_wfm:.4f} MAE={best_mae:.4f}\n")

    save_checkpoint(epoch, net, ema, optimizer, scheduler, scaler,
                    best_val_loss, best_epoch,
                    train_losses, val_losses, lr_list,
                    mae_list, sm_list, em_list, wfm_list, bfm_list)

    # ── Plots ────────────────────────────────────────────────
    os.makedirs("figures", exist_ok=True)
    x = list(range(1, len(train_losses)+1))

    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(x, train_losses, label="Train",      color='#1f77b4', marker='o', markersize=3, lw=2)
    ax.plot(x, val_losses,   label="Val (EMA)",  color='#ff7f0e', marker='o', markersize=3, lw=2)
    if best_epoch>0:
        ax.axvline(x=best_epoch, ls='--', color='red', label=f'Best({best_epoch})')
        ax.scatter(best_epoch, val_losses[best_epoch-1], color='red', zorder=5, s=50)
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.grid(True, alpha=0.3); ax.legend()
    plt.title(f"Loss Curve — Best={best_epoch}"); plt.tight_layout()
    plt.savefig("figures/loss_curve.png", dpi=150); plt.close()

    fig, axes = plt.subplots(1, 2, figsize=(14,4))
    for vals,lbl,mk in [(wfm_list,"wFm",'o'),(bfm_list,"bFm",'s'),
                         (sm_list,"Sm",'^'),(em_list,"Em",'d')]:
        axes[0].plot(x, vals, label=lbl, lw=2, marker=mk, markersize=3)
    axes[0].set_ylim(0,1); axes[0].grid(True, alpha=0.3)
    axes[0].legend(loc="lower right", fontsize=9)
    axes[0].set_title("Segmentation Metrics (↑)  [EMA val]")
    axes[1].plot(x, mae_list, color='tomato', lw=2, marker='o', markersize=3)
    axes[1].grid(True, alpha=0.3); axes[1].set_title("MAE (↓)  [EMA val]")
    plt.suptitle(f"BCE×{LOSS_W_BCE} | SSIM×{LOSS_W_SSIM} | IOU×{LOSS_W_IOU} | effBS={EFFECTIVE_BATCH_SIZE}", fontsize=11)
    plt.tight_layout()
    plt.savefig("figures/metrics_curve.png", dpi=150, bbox_inches='tight'); plt.close()

    fig, ax = plt.subplots(figsize=(7,4))
    ax.plot(x, lr_list, lw=2, marker='o', markersize=3)
    ax.set_yscale('log'); ax.grid(True)
    plt.title("LR Schedule (per-step cosine)"); plt.tight_layout()
    plt.savefig("figures/lr_schedule.png", dpi=150); plt.close()

print(f"\nBest epoch: {best_epoch} | Val loss: {best_val_loss:.6f}")
print(f"Best WFm: {best_wfm:.4f} | Best MAE: {best_mae:.4f}")
print("--- Training Done! ---")


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:44: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


Train: 6000 | Val: 1500
Effective batch: 32 (bs=4 × accum=8)
Loss weights: BCE×1.0 | SSIM×1.0 | IOU×1.0


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 165MB/s]


Model on: cuda:0
[Checkpoint] Not found — training from scratch.

--- Training from epoch 1/30 ---
GPU: 13.8GB free / 14.6GB total


Epoch 1/30 [train]:   3%|▎         | 42/1500 [01:42<48:27,  1.99s/it]  

Evaluate:

In [ ]:
# ============================================================
#  DUTS-TE EVALUATION SCRIPT
#  Run this AFTER training to evaluate on test set
#  Usage: python basnet_evaluate.py
# ============================================================

import torch
import os
import glob
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.amp
from tqdm import tqdm
from scipy.ndimage import (
    distance_transform_edt,
    binary_erosion,
    binary_dilation,
    generate_binary_structure,
)

from data_loader import (
    RescaleT, ToTensorLab, SalObjDataset
)
from model import BASNet

# ============================================================
#  EVALUATION METRICS (same as basnet_train.py - DO NOT CHANGE)
# ============================================================

def compute_mae(pred, gt):
    pred_np = pred.squeeze().cpu().float().numpy().astype(np.float64)
    gt_np   = gt.squeeze().cpu().float().numpy().astype(np.float64)
    return float(np.mean(np.abs(pred_np - gt_np)))

def _ssim_region(pred, gt):
    x = pred.mean()
    y = gt.mean()
    sigma_x  = ((pred - x) ** 2).mean()
    sigma_y  = ((gt  - y) ** 2).mean()
    sigma_xy = ((pred - x) * (gt - y)).mean()
    num = 4 * x * y * sigma_xy
    den = (x ** 2 + y ** 2) * (sigma_x + sigma_y) + 1e-8
    if num == 0:
        return 1.0 if den == 0 else 0.0
    return float(num / den)

def _centroid(gt):
    rows, cols = gt.shape
    if gt.sum() == 0:
        return cols // 2, rows // 2
    j_grid, i_grid = np.meshgrid(np.arange(cols), np.arange(rows))
    total = gt.sum()
    X = int((j_grid * gt).sum() / total)
    Y = int((i_grid * gt).sum() / total)
    return X, Y

def _divide(arr, X, Y):
    return arr[:Y, :X], arr[:Y, X:], arr[Y:, :X], arr[Y:, X:]

def _So(pred, gt):
    fg = pred * gt
    bg = (1 - pred) * (1 - gt)
    u  = gt.mean()
    return u * _ssim_region(fg, gt) + (1 - u) * _ssim_region(bg, 1 - gt)

def _Sr(pred, gt):
    X, Y  = _centroid(gt)
    h, w  = gt.shape
    total = h * w
    score = 0.0
    for gp, pp in zip(_divide(gt, X, Y), _divide(pred, X, Y)):
        w_i = gp.size / total
        score += w_i * _ssim_region(pp, gp)
    return score

def compute_smeasure(pred, gt, alpha=0.5):
    pred_np = pred.squeeze().cpu().float().numpy().astype(np.float64)
    gt_np   = (gt.squeeze().cpu().float().numpy() >= 0.5).astype(np.float64)
    y = gt_np.mean()
    if y == 0:
        score = 1.0 - pred_np.mean()
    elif y == 1:
        score = pred_np.mean()
    else:
        score = alpha * _So(pred_np, gt_np) + (1 - alpha) * _Sr(pred_np, gt_np)
    return float(max(0.0, score))

def compute_emeasure(pred, gt):
    pred_np = pred.squeeze().cpu().float().numpy()
    gt_np = (gt.squeeze().cpu().float().numpy() >= 0.5).astype(np.float64)

    th = 2 * pred_np.mean()
    if th > 1: th = 1
    pred_bin = (pred_np >= th).astype(np.float64)

    if gt_np.max() == 0:
        enhanced = 1.0 - pred_bin
    elif gt_np.min() == 1:
        enhanced = pred_bin
    else:
        mu_p = pred_bin.mean()
        mu_g = gt_np.mean()
        dp = pred_bin - mu_p
        dg = gt_np - mu_g
        align = 2.0 * dp * dg / (dp ** 2 + dg ** 2 + 1e-8)
        enhanced = ((align + 1.0) ** 2) / 4.0

    return float(enhanced.mean())

def compute_wfmeasure(pred, gt, beta_sq=1.0):
    pred_np = pred.squeeze().cpu().float().numpy().astype(np.float64)
    gt_np   = (gt.squeeze().cpu().float().numpy() >= 0.5).astype(np.float64)

    if gt_np.max() == 0:
        return 0.0

    E = np.abs(pred_np - gt_np)
    Dst = distance_transform_edt(1 - gt_np)
    if Dst.max() > 0:
        Dst = Dst / Dst.max()

    weight = 1.0 + 5.0 * Dst
    TP = (gt_np * (1 - E) * weight).sum()
    FP = ((1 - gt_np) * E * weight).sum()

    precision = TP / (TP + FP + 1e-8)
    recall    = TP / (gt_np.sum() + 1e-8)

    Fw = (1.0 + beta_sq) * precision * recall / (beta_sq * precision + recall + 1e-8)
    return float(Fw)

def compute_boundary_fmeasure(pred, gt, threshold=0.5, beta_sq=0.3, tolerance=3):
    pred_bin = (pred.squeeze().cpu().float().numpy() >= threshold).astype(np.uint8)
    gt_bin   = (gt.squeeze().cpu().float().numpy()   >= 0.5      ).astype(np.uint8)

    struct      = generate_binary_structure(2, 1)
    pred_bd     = pred_bin ^ binary_erosion(pred_bin, struct).astype(np.uint8)
    gt_bd       = gt_bin   ^ binary_erosion(gt_bin,   struct).astype(np.uint8)

    pred_bd_dil = binary_dilation(pred_bd, iterations=tolerance)
    gt_bd_dil   = binary_dilation(gt_bd,   iterations=tolerance)

    pred_bd_sum = pred_bd.sum()
    gt_bd_sum   = gt_bd.sum()

    precision = float((pred_bd & gt_bd_dil).sum()) / (pred_bd_sum + 1e-8) \
                if pred_bd_sum > 0 else 0.0
    recall    = float((gt_bd & pred_bd_dil).sum()) / (gt_bd_sum + 1e-8) \
                if gt_bd_sum > 0 else 0.0

    Fb = (1.0 + beta_sq) * precision * recall / (beta_sq * precision + recall + 1e-8)
    return float(Fb)

# ============================================================
#  MAIN EVALUATION
# ============================================================

if __name__ == '__main__':

    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True

    # ------- Model Loading --------
    model_path = "./saved_models/basnet_bsi/basnet_epoch_best_sm.pth"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if not os.path.exists(model_path):
        print(f"[ERROR] Model not found at {model_path}")
        print("Please train the model first using basnet_train.py")
        exit(1)

    net = BASNet(3, 1).to(device)
    net.load_state_dict(torch.load(model_path, map_location=device))
    net.eval()
    print(f"[LOAD] Model loaded from {model_path}")
    print(f"[DEVICE] {device}")

    # ------- DUTS-TE Dataset --------
    test_data_dir = './validation_data/'
    test_image_dir = 'DUTS-TE/DUTS-TE-Image/'
    test_label_dir = 'DUTS-TE/DUTS-TE-Mask/'
    image_ext = '.jpg'
    label_ext = '.png'

    SIZE_TEST = 256

    test_img_name_list = glob.glob(test_data_dir + test_image_dir + '*' + image_ext)

    test_lbl_name_list = []
    for img_path in test_img_name_list:
        img_name = os.path.basename(img_path)
        imidx = os.path.splitext(img_name)[0]
        test_lbl_name_list.append(
            os.path.join(test_data_dir, test_label_dir, imidx + label_ext)
        )

    print(f"[DATA] Found {len(test_img_name_list)} test images")

    # ------- DataLoader --------
    test_dataset = SalObjDataset(
        img_name_list=test_img_name_list,
        lbl_name_list=test_lbl_name_list,
        transform=transforms.Compose([
            RescaleT(SIZE_TEST),
            ToTensorLab(flag=0)
        ])
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    # ------- Evaluation --------
    print("\n[EVAL] Evaluating on DUTS-TE...")
    total_mae = 0.0
    total_sm = 0.0
    total_em = 0.0
    total_wfm = 0.0
    total_bfm = 0.0
    total_samples = 0

    with torch.no_grad():
        test_bar = tqdm(test_loader, desc="DUTS-TE Evaluation", leave=True)
        for data in test_bar:
            inputs, labels = data['image'], data['label']
            inputs = inputs.float().to(device)
            labels = labels.float().to(device)

            with torch.amp.autocast('cuda', dtype=torch.float16):
                d0, _, _, _, _, _, _, _ = net(inputs)

            pred = d0.float()

            for b in range(pred.shape[0]):
                total_mae += compute_mae(pred[b], labels[b])
                total_sm += compute_smeasure(pred[b], labels[b])
                total_em += compute_emeasure(pred[b], labels[b])
                total_wfm += compute_wfmeasure(pred[b], labels[b])
                total_bfm += compute_boundary_fmeasure(pred[b], labels[b])
                total_samples += 1

            test_bar.set_postfix({
                "MAE": f"{total_mae / total_samples:.4f}",
                "Sm": f"{total_sm / total_samples:.3f}",
                "em": f"{total_em / total_samples:.3f}",
                "wFm": f"{total_wfm / total_samples:.3f}",
                "bFm": f"{total_bfm / total_samples:.3f}",
            })

            torch.cuda.empty_cache()

    # ------- Results --------
    avg_mae = total_mae / total_samples
    avg_sm = total_sm / total_samples
    avg_em = total_em / total_samples
    avg_wfm = total_wfm / total_samples
    avg_bfm = total_bfm / total_samples

    print("\n" + "="*60)
    print("FINAL EVALUATION RESULTS ON DUTS-TE")
    print("="*60)
    print(f"  (1) Weighted Fm  F^w : {avg_wfm:.4f}  -- Best")
    print(f"  (2) Boundary Fm  F^b : {avg_bfm:.4f}  -- Best")
    print(f"  (3) MAE          M   : {avg_mae:.4f}  (down, lower better)")
    print(f"  (4) S-measure    S_a : {avg_sm:.4f}  -- Best")
    print(f"  (5) E-measure    E_m : {avg_em:.4f}  -- Best")
    print("="*60)
    print(f"Total samples: {total_samples}")
    print("="*60 + "\n")


ModuleNotFoundError: No module named 'data_loader'